# Roboflow Fire/Smoke Dataset v2 on Kaggle

This notebook downloads the `sayed-gamall/fire-smoke-detection-yolov11` Roboflow dataset, cleans invalid YOLO rows, previews the data, trains a YOLO26 model, and evaluates it.

In [ ]:
!pip install -qU roboflow ultralytics pyyaml requests

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import random
import zipfile
import requests
import yaml
import math

import torch
import cv2
import matplotlib.pyplot as plt

from roboflow import Roboflow
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

USE_KAGGLE_SECRET = False

if USE_KAGGLE_SECRET:
    from kaggle_secrets import UserSecretsClient
    ROBOFLOW_API_KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
else:
    ROBOFLOW_API_KEY = "PASTE_YOUR_API_KEY"

WORKSPACE_ID = "sayed-gamall"
PROJECT_ID = "fire-smoke-detection-yolov11"
VERSION_NUMBER = 2
EXPORT_FORMAT = "yolo26"

ZIP_PATH = Path("/kaggle/working/rf_fire_smoke_v2.zip")
EXTRACT_DIR = Path("/kaggle/working/rf_fire_smoke_v2")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
api_url = f"https://api.roboflow.com/{WORKSPACE_ID}/{PROJECT_ID}/{VERSION_NUMBER}/{EXPORT_FORMAT}?api_key={ROBOFLOW_API_KEY}"

resp = requests.get(api_url, timeout=120)
resp.raise_for_status()
payload = resp.json()

print(payload.keys())
print(payload["export"]["link"])

In [ ]:
download_url = payload["export"]["link"]

with requests.get(download_url, stream=True, timeout=600) as r:
    r.raise_for_status()
    with open(ZIP_PATH, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("Downloaded:", ZIP_PATH)
print("Size MB:", round(ZIP_PATH.stat().st_size / (1024 * 1024), 2))

In [ ]:
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print("Extracted to:", EXTRACT_DIR)

In [ ]:
def normalize_names(names):
    if isinstance(names, list):
        return {i: name for i, name in enumerate(names)}
    return {int(k): v for k, v in names.items()}

split_hits = defaultdict(set)

for split in ["train", "valid", "val", "test"]:
    pattern = f"{split}/images"
    for img_dir in EXTRACT_DIR.rglob(pattern):
        root = img_dir.parent.parent
        if (root / split / "labels").exists():
            split_hits[root].add(split)

if not split_hits:
    raise FileNotFoundError(f"Could not detect dataset root under {EXTRACT_DIR}")

DATASET_ROOT = sorted(split_hits.items(), key=lambda x: (-len(x[1]), len(str(x[0]))))[0][0]
print("Detected dataset root:", DATASET_ROOT)

source_yaml = DATASET_ROOT / "data.yaml"
if source_yaml.exists():
    source_cfg = yaml.safe_load(source_yaml.read_text())
    class_names = normalize_names(source_cfg.get("names", {0: "smoke", 1: "fire"}))
else:
    class_names = {0: "smoke", 1: "fire"}

CLASS_NAMES = class_names
ALLOWED_CLASSES = set(CLASS_NAMES.keys())

DATA_YAML = Path("/kaggle/working/data.yaml")
cfg = {
    "path": str(DATASET_ROOT),
    "train": "train/images" if (DATASET_ROOT / "train" / "images").exists() else None,
    "val": "valid/images" if (DATASET_ROOT / "valid" / "images").exists() else ("val/images" if (DATASET_ROOT / "val" / "images").exists() else None),
    "test": "test/images" if (DATASET_ROOT / "test" / "images").exists() else None,
    "names": CLASS_NAMES,
}
cfg = {k: v for k, v in cfg.items() if v is not None}

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(DATA_YAML.read_text())

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def existing_split_names(root: Path):
    splits = []
    for name in ["train", "valid", "val", "test"]:
        if (root / name / "images").exists() and (root / name / "labels").exists():
            splits.append(name)
    return splits

SPLITS = existing_split_names(DATASET_ROOT)
print("Detected splits:", SPLITS)

In [ ]:
def clean_label_file(label_path: Path):
    raw_lines = label_path.read_text(encoding="utf-8").splitlines()
    kept = []
    removed = []

    for line_no, raw in enumerate(raw_lines, 1):
        line = raw.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) != 5:
            removed.append((line_no, line, "wrong_column_count"))
            continue

        try:
            cls = int(parts[0])
            x, y, w, h = map(float, parts[1:])
        except Exception:
            removed.append((line_no, line, "parse_error"))
            continue

        if cls not in ALLOWED_CLASSES:
            removed.append((line_no, line, "invalid_class"))
            continue

        if any(v < 0 or v > 1 for v in (x, y, w, h)):
            removed.append((line_no, line, "coords_out_of_bounds"))
            continue

        if w <= 0 or h <= 0:
            removed.append((line_no, line, "non_positive_size"))
            continue

        kept.append(f"{cls} {x} {y} {w} {h}")

    label_path.write_text("\n".join(kept), encoding="utf-8")
    return kept, removed

stats = Counter()

for split in SPLITS:
    label_dir = DATASET_ROOT / split / "labels"
    for label_path in label_dir.glob("*.txt"):
        before = [x for x in label_path.read_text(encoding="utf-8").splitlines() if x.strip()]
        kept, removed = clean_label_file(label_path)

        stats["files_seen"] += 1
        stats["rows_before"] += len(before)
        stats["rows_after"] += len(kept)
        stats["rows_removed"] += len(removed)

        if removed:
            stats["files_modified"] += 1
            for _, _, reason in removed:
                stats[f"removed_{reason}"] += 1

print(dict(stats))

In [ ]:
for split in SPLITS:
    img_dir = DATASET_ROOT / split / "images"
    lbl_dir = DATASET_ROOT / split / "labels"

    image_count = sum(1 for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS)
    total_labels = 0
    nonempty = 0
    class_counts = Counter()

    for txt in lbl_dir.glob("*.txt"):
        total_labels += 1
        lines = [x.strip() for x in txt.read_text().splitlines() if x.strip()]
        if lines:
            nonempty += 1
        for line in lines:
            cls = int(line.split()[0])
            class_counts[cls] += 1

    print(f"\n[{split}]")
    print("images:", image_count)
    print("label files:", total_labels)
    print("nonempty labels:", nonempty)
    print("class counts:", class_counts)

In [ ]:
def show_random_nonempty(split="train", n=12):
    img_dir = DATASET_ROOT / split / "images"
    lbl_dir = DATASET_ROOT / split / "labels"

    candidates = []
    for txt in lbl_dir.glob("*.txt"):
        if txt.read_text().strip():
            img = None
            for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
                p = img_dir / f"{txt.stem}{ext}"
                if p.exists():
                    img = p
                    break
            if img:
                candidates.append((img, txt))

    picks = random.sample(candidates, min(n, len(candidates)))

    cols = 3
    rows = math.ceil(len(picks) / cols)
    plt.figure(figsize=(16, 5 * rows))

    for i, (img_path, txt_path) in enumerate(picks, 1):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        for line in txt_path.read_text().splitlines():
            if not line.strip():
                continue
            cls, xc, yc, bw, bh = map(float, line.split())
            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)
            label = CLASS_NAMES[int(cls)]
            color = (0, 170, 255) if label.lower() == "smoke" else (255, 80, 80)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, label, (x1, max(20, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        plt.subplot(rows, cols, i)
        plt.imshow(img)
        plt.title(img_path.name)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_random_nonempty("train", 12)

In [ ]:
MODEL_NAME = "yolo26n.pt"
RUN_NAME = "rf_fire_smoke_v2_yolo26n"
EPOCHS = 20
IMGSZ = 512
BATCH = 16
TRAIN_FRACTION = 0.2

model = YOLO(MODEL_NAME)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=4,
    cache="disk",
    pretrained=True,
    optimizer="auto",
    cos_lr=True,
    patience=10,
    close_mosaic=10,
    project="/kaggle/working/runs/detect",
    name=RUN_NAME,
    exist_ok=True,
    seed=SEED,
    deterministic=False,
    amp=False,
    save=True,
    val=True,
    plots=True,
    fraction=TRAIN_FRACTION,
)

In [ ]:
best_path = Path(results.save_dir) / "weights" / "best.pt"
print("Best model:", best_path)

In [ ]:
best_model = YOLO(str(best_path))

val_split = "test" if "test" in SPLITS else ("valid" if "valid" in SPLITS else "val")

metrics = best_model.val(
    data=str(DATA_YAML),
    split=val_split,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=4,
    plots=True
)

print(metrics.results_dict)